# TCL-v0 Kaggle Ablation Smoke Test

Purpose: run the next TCL-v0 ablation checkpoint on Kaggle before any larger scale-up.

This notebook follows `TCL-v0-Ablation-Plan.md` and `TCL-v0-Kaggle-Ablation-Runbook.md`. It is a smoke-test protocol, not a final evidence report.

## Kaggle Requirements

- GPU accelerator enabled.
- Internet enabled for repository clone and dataset access.
- Kaggle-hosted Gemma 2B-it model attached, or `MODEL` changed to an accessible Hugging Face model ID.
- The smoke test should complete before any 1,000-example ablation run.

In [ ]:
!git clone https://github.com/awabmoha/truth-calibration-layer.git
%cd truth-calibration-layer/tcl_experiments
!python -m pip install -r requirements.txt
!python scripts/check_runtime.py

## Prepare Benchmark Data

The ablation smoke test uses the fixed TriviaQA-1000 split and limits execution to the first 100 records by default.

In [ ]:
!python scripts/prepare_triviaqa_subset.py --limit 1000 --seed 42

## Run 100-Example Ablation Smoke Test

This cell runs 3 pooling methods by 3 layer positions and then computes raw-only and hidden-state ablation metrics.

In [ ]:
%env MODEL=/kaggle/input/models/google/gemma/transformers/2b-it/3
%env DATASET=triviaqa_rc_nocontext_validation_1000
%env QUESTIONS=data/benchmarks/triviaqa/triviaqa_rc_nocontext_validation_1000.csv
%env SPLITS=data/benchmarks/triviaqa/triviaqa_rc_nocontext_validation_1000_splits.csv
%env RUN_PREFIX=ablation-smoke-gemma-triviaqa100
%env LIMIT=100
%env DEVICE=auto
%env DTYPE=auto
!bash scripts/run_free_gpu_ablation_smoke.sh

## Inspect Smoke-Test Outputs

A successful smoke test creates nine run folders plus an ablation summary folder. One-class test splits may demonstrate pipeline execution but should not be treated as research evidence.

In [ ]:
!ls -d runs/ablation-smoke-gemma-triviaqa100-* | sort
!python - <<'PY'
import json
from pathlib import Path
p = Path('runs/ablation-smoke-gemma-triviaqa100-ablation_summary/ablation_metrics.json')
print(json.dumps(json.loads(p.read_text()), indent=2)[:4000] if p.exists() else 'missing ablation_metrics.json')
PY

## Package Artifacts

The artifact zip should be preserved for local import and review.

In [ ]:
!zip -r runs/ablation-smoke-gemma-triviaqa100_artifact.zip runs/ablation-smoke-gemma-triviaqa100-ablation_summary runs/ablation-smoke-gemma-triviaqa100-*